# DCP pilot – test run in Google Colab
Run the cells one by one from top to bottom (click ▶ on the left of each cell). Synthetic data only.

## Step 1 – Upload the package
Run the cell, click **Choose Files** and select `dcp-pilot-v1.0.0.zip` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Step 2 – Unzip and install the two required libraries

In [ ]:
!unzip -o -q dcp-pilot-v1.0.0.zip
%cd dcp-pilot
!pip install -q -r requirements.txt
!ls

## Step 3 – Run the complete pilot
The last line must say: `RESULT: all checks behaved as expected`

In [ ]:
!python src/run_pilot.py

## Step 4 – See the cement passport
The first QR holds a short text summary of the batch (any phone shows it, no internet needed). Below it is the passport page; the QR printed on that page opens the online copy of the page.

In [ ]:
from IPython.display import HTML, Image, display
display(Image('output/qr_DCP-EX-2026-000123_summary.png', width=260))
display(HTML(open('output/passport_DCP-EX-2026-000123.html', encoding='utf-8').read()))

# To save the page on your computer, uncomment the next two lines:
# from google.colab import files
# files.download('output/passport_DCP-EX-2026-000123.html')

## Step 5 – Run the automated tests
The end must say `OK` (one or two tests may be `skipped` – that is normal).

In [ ]:
!python -m unittest discover -s tests -v

## Step 6 – Role-dependent views (public / engineer / auditor)

In [ ]:
!python src/passport_report.py examples/batch-valid.json public
!python src/passport_report.py examples/batch-valid.json engineer

## Step 7 – Try it yourself: break the record and see it rejected
Change the numbers in this cell and run it again. Example: chromium VI = 3.1 (limit 2 mg/kg), or clinker ratio = 85 for CEM II/B-V (allowed 65–79).

In [ ]:
import json, subprocess, sys
record = json.load(open('examples/batch-valid.json'))

# ---- change values here ----
record['safety']['solubleChromiumVI'] = 3.1
record['composition']['clinkerRatio'] = 74.0
# ----------------------------

json.dump(record, open('my_test.json', 'w'), indent=2)
print(subprocess.run([sys.executable, 'src/validate_passport.py', 'my_test.json'], capture_output=True, text=True).stdout)

## Step 8 – Offline simulation of the MQTT A3 update

In [ ]:
!python src/simulate_a3_update.py examples/batch-valid.json